# 7.3 · 偏差-方差权衡 / Bias-Variance Trade-off

> **课程定位 / Where this fits**
> 第 3 课，**Part 7 · 模型评估与优化**。
> Lesson 3, **Part 7 · Model Evaluation & Tuning**.
>
> 4.3 用多项式直观看过偏差方差。这一课把它**讲透并形式化**：误差的三段分解、用学习曲线/验证曲线**诊断**模型到底是高偏差还是高方差，以及对症下药的**完整对策清单**。这是机器学习**最核心的思维框架**——几乎所有调优决策都基于它。
> 4.3 visualized bias-variance with polynomials. This lesson makes it **rigorous**: the three-part error decomposition, **diagnosing** high-bias vs high-variance via learning/validation curves, and a **complete fix checklist**. This is ML's **central mental model** — almost every tuning decision rests on it.
>
> 💼 **实战/面试视角**："偏差方差是什么 / 怎么判断欠拟合还是过拟合 / 各自怎么救" 是 ML 面试**最核心**的概念题, 几乎必问。
> 💼 **Practical/interview angle:** "what is bias-variance / underfit vs overfit / how to fix each" — *the* core ML concept, near-guaranteed.

> 📐 **符号约定 / Notation**
> - 偏差² bias² —— 模型平均预测偏离真值的程度 / how far average prediction is from truth
> - 方差 variance —— 模型随训练数据变化的波动 / variability across training sets
> - 不可约噪声 irreducible noise —— 数据本身的随机性 / inherent randomness

> 💡 **面试相关 / Interview-relevant**
> - "偏差-方差分解（公式三部分）"（出镜率 ★★★★★）
> - "高偏差 vs 高方差的表现"（★★★★★）
> - "学习曲线 vs 验证曲线分别看什么"（★★★★★）
> - "高偏差/高方差分别怎么救"（★★★★★）
> - "为什么不能同时把两者降到最低"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解误差 = 偏差² + 方差 + 不可约噪声。
   Understand error = bias² + variance + irreducible noise.
2. 用实验**实测**偏差与方差随复杂度的变化。
   Empirically measure bias and variance versus complexity.
3. 用**验证曲线**和**学习曲线**诊断模型。
   Diagnose models with validation and learning curves.
4. 掌握高偏差/高方差的**对策清单**。
   Master the fix checklist for high-bias / high-variance.

## 目录 / TOC
1. [先建直觉 + 三段分解 ⭐](#1)
2. [📈 实测偏差与方差 ⭐](#2)
3. [验证曲线：诊断复杂度 ⭐](#3)
4. [学习曲线：诊断数据是否够 ⭐](#4)
5. [对策清单 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 三段分解 ⭐ / Intuition & the Decomposition

打靶比喻最直观：**偏差**是"瞄准是否系统性偏离靶心"，**方差**是"每次射击是否散得很开"。
The dartboard analogy: **bias** is "is your aim systematically off-center", **variance** is "do your shots scatter widely".
- **高偏差**：枪枪打偏但很集中（模型太简单，系统性错，**欠拟合**）。
  **High bias:** tightly grouped but off-target (too simple, systematic error, **underfitting**).
- **高方差**：平均在靶心但散得很开（模型太复杂，对训练数据的随机扰动过度敏感，**过拟合**）。
  **High variance:** centered on average but widely scattered (too complex, over-sensitive to training noise, **overfitting**).

数学上，对一个新点的**期望预测误差**可精确分解成三部分（面试要点）：
Mathematically, the expected prediction error at a new point decomposes exactly into three parts (interview point):

$$\mathbb E[(y-\hat f)^2] = \underbrace{(\mathbb E[\hat f]-f)^2}_{\text{偏差}^2} + \underbrace{\mathbb E[(\hat f-\mathbb E[\hat f])^2]}_{\text{方差}} + \underbrace{\sigma^2}_{\text{不可约噪声}}$$

- **偏差²**：模型的"平均预测"离真值多远（简单模型大）。
  **bias²:** how far the model's *average* prediction is from the truth (large for simple models).
- **方差**：换一批训练数据，预测会变多少（复杂模型大）。
  **variance:** how much the prediction changes with a different training set (large for complex models).
- **不可约噪声 $\sigma^2$**：数据本身的随机性，**任何模型都消不掉**（这是误差的下限）。
  **irreducible noise $\sigma^2$:** inherent randomness, **unremovable by any model** (the error floor).

**关键权衡**：复杂度↑ → 偏差↓ 但方差↑。所以**不能同时把两者降到最低**，最优在中间。
**The trade-off:** complexity↑ → bias↓ but variance↑. So you **can't minimize both at once**; the optimum is in the middle.


<a id="2"></a>
## 2. 实测偏差与方差 ⭐ / Measuring Bias and Variance

把抽象公式变成可见的曲线：对每个模型复杂度，**重复采样多次训练**，在固定测试点上记录所有预测，然后算偏差²（平均预测离真值）和方差（预测之间的散布）。会看到经典的"偏差降、方差升、总误差 U 形"。
Turn the abstract formula into visible curves: for each complexity, **resample and retrain many times**, record predictions at fixed test points, then compute bias² (average prediction vs truth) and variance (spread among predictions). You'll see the classic "bias down, variance up, total error U-shaped".


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

def true_f(x): return np.sin(1.5*x)          # 真实函数 / true function
sigma = 0.3                                   # 噪声水平(=不可约误差的来源)
x_test = np.linspace(0, 4, 60).reshape(-1, 1)
y_test_true = true_f(x_test).ravel()

degrees = range(1, 12)
n_repeats = 200
bias2, var, total = [], [], []
for deg in degrees:
    preds = np.zeros((n_repeats, len(x_test)))
    for r in range(n_repeats):
        xr = rng.uniform(0, 4, 50)                          # 每次重采样训练集
        yr = true_f(xr) + rng.normal(0, sigma, 50)
        m = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()).fit(xr.reshape(-1,1), yr)
        preds[r] = m.predict(x_test)
    mean_pred = preds.mean(0)                                # 这个复杂度的"平均预测"
    bias2.append(np.mean((mean_pred - y_test_true)**2))      # 偏差² = 平均预测离真值
    var.append(np.mean(preds.var(0)))                        # 方差 = 不同采样间的散布
    total.append(bias2[-1] + var[-1])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(list(degrees), bias2, "o-", label="偏差² bias²(简单模型大)")
ax.plot(list(degrees), var, "s-", label="方差 variance(复杂模型大)")
ax.plot(list(degrees), total, "^-", lw=2, label="总误差 = bias²+var")
ax.axhline(sigma**2, color="gray", ls=":", label=f"不可约噪声 σ²={sigma**2:.2f}(下限)")
best = list(degrees)[int(np.argmin(total))]
ax.axvline(best, color="r", ls="--", alpha=0.6, label=f"最优复杂度={best}")
ax.set_xlabel("多项式阶数(复杂度)"); ax.set_ylabel("error"); ax.legend(fontsize=8)
ax.set_title("偏差-方差权衡: 总误差 U 形, 最优在中等复杂度")
plt.tight_layout(); plt.show()
print(f"低阶: 高偏差(欠拟合); 高阶: 高方差(过拟合); 总误差最低在阶数≈{best}")
print(f"不可约噪声 σ²={sigma**2:.2f} 是误差地板 — 再好的模型也降不到它以下")


<a id="3"></a>
## 3. 验证曲线：诊断复杂度 ⭐ / Validation Curve

实测偏差方差需要知道真值，现实中没有。**验证曲线**用 train vs CV 分数间接诊断：横轴是某个复杂度超参，看两条曲线。这是诊断**"复杂度选对了吗"**的标准工具。
Measuring bias/variance needs the ground truth, unavailable in practice. The **validation curve** diagnoses indirectly via train vs CV scores: x-axis is a complexity hyperparameter, watch both curves. The standard tool for "is the complexity right".
- **两条都低且接近** → 高偏差（欠拟合），复杂度不够。
  **Both low and close** → high bias (underfit), not complex enough.
- **train 高、CV 低、有大缺口** → 高方差（过拟合），复杂度过头。
  **train high, CV low, big gap** → high variance (overfit), too complex.


In [ ]:
from sklearn.model_selection import validation_curve

# 生成一份固定数据集做诊断 / one fixed dataset
x = rng.uniform(0, 4, 120); y = true_f(x) + rng.normal(0, sigma, 120)
X = x.reshape(-1, 1)
degs = np.arange(1, 13)
train_s, val_s = validation_curve(
    make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression()),
    X, y, param_name="polynomialfeatures__degree", param_range=degs, cv=5, scoring="r2")

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(degs, train_s.mean(1), "o-", label="train R²")
ax.plot(degs, val_s.mean(1), "s-", label="CV R²")
ax.axvspan(1, 2, alpha=0.08, color="blue"); ax.text(1.2, 0.1, "高偏差区\n欠拟合", fontsize=8)
ax.axvspan(9, 12, alpha=0.08, color="red"); ax.text(9.5, 0.1, "高方差区\n过拟合", fontsize=8)
ax.set_xlabel("多项式阶数(复杂度)"); ax.set_ylabel("R²"); ax.set_ylim(-0.1, 1.05); ax.legend()
ax.set_title("验证曲线: 左=高偏差(都低), 右=高方差(train高CV低有缺口)")
plt.tight_layout(); plt.show()
print("左端(阶数小): train/CV 都低 → 高偏差; 右端: train↑ 但 CV↓ 缺口大 → 高方差")
print("选 CV 峰值对应的复杂度")


<a id="4"></a>
## 4. 学习曲线：诊断数据是否够 ⭐ / Learning Curve

验证曲线诊断"复杂度"，**学习曲线诊断"数据量够不够"**：横轴是训练样本数，看 train 和 CV 怎么随数据增多而变。它直接回答一个高价值的实战问题——**"我该去搞更多数据，还是该换模型/调正则？"**
The validation curve diagnoses complexity; the **learning curve diagnoses whether you have enough data**: x-axis is training-set size, watch train and CV evolve. It directly answers a high-value question — **"should I gather more data, or change the model/regularization?"**
- **高偏差**：train、CV 都低且**早早收敛到一起**。→ **加数据没用**，要更复杂的模型/更多特征。
  **High bias:** both low and **converge early**. → **More data won't help**; need complexity/features.
- **高方差**：train 高、CV 低、**有持续的大缺口**。→ **加数据能缩小缺口**（或降复杂度/加正则）。
  **High variance:** train high, CV low, **persistent gap**. → **More data closes the gap** (or reduce complexity / regularize).


In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, deg, title in [(axes[0], 1, "高偏差 high-bias (阶=1)"),
                       (axes[1], 11, "高方差 high-variance (阶=11)")]:
    sizes, tr, va = learning_curve(
        make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression()),
        X, y, cv=5, scoring="r2", train_sizes=np.linspace(0.15, 1.0, 8))
    ax.plot(sizes, tr.mean(1), "o-", label="train")
    ax.plot(sizes, va.mean(1), "s-", label="CV")
    ax.set_xlabel("训练样本数 #train"); ax.set_ylabel("R²"); ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()
print("左(高偏差): train/CV 都低且早收敛 → 加数据没用, 要更复杂模型/更多特征")
print("右(高方差): train 高 CV 低有大缺口 → 加数据能缩小缺口(或降复杂度/加正则)")


<a id="5"></a>
## 5. 对策清单 + 小结 ⭐ / Fix Checklist & Summary

诊断出问题后，对症下药（这是本课最实用的产出）：
After diagnosis, apply the right fix (the most practical takeaway):

| 问题 / Problem | 表现 / Signature | 对策 / Fixes |
|---|---|---|
| **高偏差（欠拟合）** | train、CV 都差、早收敛 | 更复杂模型 / 加特征(特征工程 3.6) / 减正则 / 多项式/核 / 训练更久 |
| **高方差（过拟合）** | train≫CV、大缺口 | 加数据 / 加正则(L1/L2 4.4-4.5) / 减复杂度 / 降维 / 集成(bagging 4.12) / 早停 / dropout |

> ⚠️ **注意**：加数据**只对高方差有用**，对高偏差无效——这是面试常考的"陷阱选择题"。先诊断，再决定。
> ⚠️ **Note:** more data **helps only high variance**, not high bias — a common interview trap. Diagnose first, then decide.

```
误差 = 偏差²(模型太简单, 系统偏离) + 方差(模型太复杂, 对数据敏感) + 不可约噪声(消不掉的下限)
权衡: 复杂度↑ → 偏差↓ 方差↑; 总误差 U 形, 不能同时最小化两者
诊断: 验证曲线(横轴=复杂度): 都低=高偏差, train高CV低有缺口=高方差
      学习曲线(横轴=样本数): 早收敛=高偏差(加数据没用), 大缺口=高方差(加数据有用)
对策: 高偏差→加复杂度/特征/减正则; 高方差→加数据/加正则/减复杂度/集成/早停
```

### 💡 面试速查 / Interview cheat-sheet
1. **误差 = 偏差² + 方差 + 不可约噪声**; 后者是任何模型的下限。
   Error = bias² + variance + irreducible noise; the last is the floor for any model.
2. **高偏差=欠拟合(都差), 高方差=过拟合(train≫CV)**。
   High bias = underfit (both poor); high variance = overfit (train≫CV).
3. **验证曲线诊断复杂度, 学习曲线诊断数据量**。
   Validation curve diagnoses complexity; learning curve diagnoses data quantity.
4. **加数据只救高方差, 救不了高偏差**(经典陷阱)。
   More data helps only high variance, not high bias (classic trap).
5. **不能同时最小化偏差和方差**, 最优在中等复杂度。
   You can't minimize both; the optimum is moderate complexity.

### 下一节 / Next
**7.4 交叉验证策略**——诊断和评估都依赖"怎么划分数据"。系统讲 K-fold/分层/分组/时序/嵌套 CV, 巩固 3.10 并补全。
**7.4 CV Strategies** — diagnosis and evaluation both depend on how you split. Systematic coverage of K-fold/stratified/group/time-series/nested CV, consolidating and completing 3.10.
